In [ ]:
from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import DecimalType

bronze = spark.read.table("bronze_salesorderheader")

required_columns = {
    "SalesOrderID",
    "OrderDate",
    "CustomerID",
    "TerritoryID",
    "OnlineOrderFlag",
    "SubTotal",
    "TaxAmt",
    "Freight",
    "load_date",
    "source_system",
    "ingestion_timestamp",
}

missing_columns = required_columns.difference(bronze.columns)

if missing_columns:
    raise ValueError(
        f"Colonnes Bronze manquantes : {sorted(missing_columns)}"
    )

null_key_count = (
    bronze
    .filter(
        col("SalesOrderID").isNull()
        | col("OrderDate").isNull()
        | col("CustomerID").isNull()
    )
    .count()
)

if null_key_count > 0:
    raise ValueError(
        f"{null_key_count} ligne(s) ont une clé obligatoire nulle."
    )

duplicate_key_count = (
    bronze
    .groupBy("SalesOrderID")
    .count()
    .filter(col("count") > 1)
    .count()
)

if duplicate_key_count > 0:
    raise ValueError(
        f"{duplicate_key_count} SalesOrderID dupliqué(s)."
    )

silver = (
    bronze
    .select(
        col("SalesOrderID").cast("long").alias("sales_order_id"),
        col("OrderDate").cast("date").alias("order_date"),
        col("CustomerID").cast("long").alias("customer_id"),
        col("TerritoryID").cast("long").alias("territory_id"),
        col("OnlineOrderFlag").cast("boolean").alias("is_online_order"),
        col("SubTotal")
        .cast(DecimalType(19, 4))
        .alias("subtotal_amount"),
        col("TaxAmt")
        .cast(DecimalType(19, 4))
        .alias("tax_amount"),
        col("Freight")
        .cast(DecimalType(19, 4))
        .alias("freight_amount"),
        col("load_date").cast("date").alias("load_date"),
        col("source_system"),
        col("ingestion_timestamp"),
    )
    .withColumn(
        "silver_processed_timestamp",
        current_timestamp(),
    )
)

(
    silver.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("silver_salesorderheader")
)

print(f"Lignes Silver : {silver.count()}")
print("Table créée : silver_salesorderheader")

In [ ]:
invalid_amounts = silver.filter(
    (col("subtotal_amount") < 0)
    | (col("tax_amount") < 0)
    | (col("freight_amount") < 0)
).count()

null_load_dates = silver.filter(
    col("load_date").isNull()
).count()

if invalid_amounts > 0:
    raise ValueError("Des montants négatifs sont présents.")

if null_load_dates > 0:
    raise ValueError("Des load_date sont nulles.")

print(f"Montants invalides : {invalid_amounts}")
print(f"load_date nulles : {null_load_dates}")
print("Contrôles Silver Fabric réussis.")